# AI-Driven Coating Formulation & Property Optimization
### Computational Materials Science Engineering Project

**Project Objective:**
Develop a machine-learning system that models the non-linear relationship between coating formulation parameters (resin, pigment, solvent, additives, curing agent, baking temperature, baking time) and final film properties (hardness, adhesion, flexibility, gloss, drying time), executes instantaneous forward predictions, and performs constrained inverse design to recommend optimal recipes for user-defined targets.

> **SCIENTIFIC INTEGRITY NOTICE:**
> The active dataset is a clearly designated **DEMONSTRATION DATASET** constructed according to polymer coating science principles. It serves to test the software pipeline, verify convergence of the optimization algorithms, and validate the interactive dashboard. Replace `data/coating_data.csv` with real experimental laboratory data when available.


In [ ]:
# 1. Core Library Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src modules are discoverable
if "." not in sys.path:
    sys.path.insert(0, ".")

from src.data_preprocessing import (
    load_dataset,
    validate_coating_data,
    print_validation_report,
    preprocess_data,
    split_data,
    INPUT_FEATURES,
    FORMULATION_INPUTS,
    PROCESS_INPUTS,
    TARGET_PROPERTIES,
    FEATURE_BOUNDS,
    TARGET_BOUNDS
)
from src.model_training import (
    train_and_compare_models,
    save_model_bundle,
    load_model_bundle
)
from src.prediction import (
    predict_properties,
    print_prediction_table
)
from src.optimization import (
    inverse_design,
    print_inverse_design_results
)
from src.evaluation import (
    compute_property_metrics_table,
    plot_actual_vs_predicted,
    plot_residuals,
    compute_permutation_feature_importance,
    plot_feature_importance,
    generate_tradeoff_candidates,
    compute_pareto_frontier,
    evaluate_experimental_validation
)
from src.visualization import (
    plot_feature_distributions,
    plot_correlation_heatmap,
    plot_coating_domain_relationships,
    plot_hardness_flexibility_tradeoff,
    plot_ai_vs_experimental_comparison
)

print("All dependencies and modules imported successfully.")


In [ ]:
# 2. Ingest Dataset
data_path = os.path.join("data", "coating_data.csv")
df_raw = load_dataset(data_path)

print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("\nFirst 5 Rows:")
df_raw.head()


In [ ]:
# 3. Comprehensive Data Integrity & Validation Audit
# Audits schema completeness, datatypes, missing values, duplicates, physical bounds,
# and checks that formulation components strictly satisfy mass conservation: sum(wt%) == 100%
validation_report = validate_coating_data(df_raw, strict_mass_balance=True)
print_validation_report(validation_report)


In [ ]:
# 4. Exploratory Data Analysis (EDA): Feature & Target Distributions
fig_dist = plot_feature_distributions(df_raw)
plt.show()


In [ ]:
# 5. EDA: Pearson Correlation Matrix Heatmap
fig_corr = plot_correlation_heatmap(df_raw)
plt.show()


In [ ]:
# 6. EDA: Core Physical & Chemical Domain Relationships
# 1. Curing agent content vs. Hardness (Crosslink density)
# 2. Resin content vs. Flexibility (Polymer chain mobility)
# 3. Pigment content vs. Gloss (Micro-roughness scattering)
# 4. Curing temperature vs. Drying time (Thermal reaction kinetics)
# 5. Curing temperature vs. Adhesion (Optimal cure vs over-cure thermal stress)
# 6. Hardness vs. Flexibility (The classic engineering trade-off)
fig_rel = plot_coating_domain_relationships(df_raw)
plt.show()


In [ ]:
# 7. Data Preprocessing & Leak-Free Train/Test Split (80% Train, 20% Test)
clean_df, prep_log = preprocess_data(df_raw, handle_missing="median", drop_duplicates=True)
X_train, X_test, y_train, y_test = split_data(clean_df, test_size=0.20, random_state=42)

print(f"Training partition : {X_train.shape[0]} samples (X: {X_train.shape[1]}, y: {y_train.shape[1]})")
print(f"Testing partition  : {X_test.shape[0]} samples (X: {X_test.shape[1]}, y: {y_test.shape[1]})")


In [ ]:
# 8. Multi-Output Machine Learning Model Benchmarking
# Trains Random Forest, Extra Trees, Gradient Boosting, and Ridge Regression Baseline.
# Evaluates R2, MAE, and RMSE on the unseen test partition.
best_name, best_model, comparison_df, detailed_results = train_and_compare_models(
    X_train, y_train, X_test, y_test, TARGET_PROPERTIES, random_state=42
)

print("\nModel Comparison Benchmark Table (Unseen Test Set):")
comparison_df


In [ ]:
# 9. In-Depth Evaluation of Selected Model
y_test_pred = best_model.predict(X_test)
metrics_table = compute_property_metrics_table(y_test, y_test_pred, TARGET_PROPERTIES)

print(f"Detailed Test Metrics for '{best_name}':")
metrics_table


In [ ]:
# 10. Actual vs. Predicted Parity Plots & Residual Diagnostics
fig_parity = plot_actual_vs_predicted(y_test, y_test_pred, TARGET_PROPERTIES)
plt.show()

fig_res = plot_residuals(y_test, y_test_pred, TARGET_PROPERTIES)
plt.show()


In [ ]:
# 11. Permutation Feature Importance Across Coating Properties
importance_by_target = compute_permutation_feature_importance(
    best_model, X_test, y_test, INPUT_FEATURES, TARGET_PROPERTIES, n_repeats=15, random_state=42
)

fig_fi = plot_feature_importance(importance_by_target)
plt.show()


In [ ]:
# 12. Forward Property Prediction for Candidate Formulations
# Test a custom recipe:
sample_formulation = {
    "Resin_Content": 46.0,
    "Pigment_Content": 23.5,
    "Solvent_Content": 21.0,
    "Additive_Content": 3.0,
    "Curing_Agent_Content": 6.5,
    "Curing_Temperature": 115.0,
    "Curing_Time": 40.0
}

forward_preds = predict_properties(sample_formulation, model_bundle={"model": best_model, "feature_names": INPUT_FEATURES, "target_names": TARGET_PROPERTIES})
print_prediction_table(sample_formulation, forward_preds)


### 13. Interactive Inverse Design Cell
**Instructions for Live Demonstration:**
Modify the desired target values below (e.g., change `Hardness` or `Flexibility`) and re-run this cell. The Differential Evolution algorithm will automatically calculate an updated, mass-balanced recipe.


In [ ]:
# User-Specified Target Properties
targets = {
    "Hardness": 85.0,
    "Adhesion": 78.0,
    "Flexibility": 40.0,
    "Gloss": 82.0,
    "Drying_Time": 45.0
}

# Optional weights (higher weight = higher optimization priority)
weights = {
    "Hardness": 1.5,
    "Adhesion": 1.0,
    "Flexibility": 1.5,
    "Gloss": 1.0,
    "Drying_Time": 1.0
}

print(f"Running Inverse Design for targets: {targets}...")
opt_results = inverse_design(
    target_properties=targets,
    property_weights=weights,
    model_bundle={"model": best_model, "feature_names": INPUT_FEATURES, "target_names": TARGET_PROPERTIES},
    popsize=20,
    maxiter=200,
    random_state=42
)

print_inverse_design_results(opt_results)


In [ ]:
# 14. Hardness vs. Flexibility Trade-off & Pareto Frontier Analysis
candidates_df = generate_tradeoff_candidates(
    model_bundle={"model": best_model, "feature_names": INPUT_FEATURES, "target_names": TARGET_PROPERTIES},
    n_samples=350,
    random_state=42
)
pareto_df = compute_pareto_frontier(candidates_df, x_col="Flexibility", y_col="Hardness")

highlighted_coords = {
    "Flexibility": opt_results["predicted_properties"]["Flexibility"],
    "Hardness": opt_results["predicted_properties"]["Hardness"]
}

fig_tradeoff = plot_hardness_flexibility_tradeoff(
    candidates_df,
    pareto_df,
    highlighted_point=highlighted_coords
)
plt.show()


In [ ]:
# 15. Bench Experimental Validation & Error Audit
# Enter actual bench measurements from a physical wet-lab trial of the optimized recipe:
experimental_results = {
    "Hardness": 86.2,      # Lab measurement (Shore D / Koenig)
    "Adhesion": 77.0,      # Lab measurement (Cross-hatch index)
    "Flexibility": 38.5,   # Lab measurement (Mandrel bend index)
    "Gloss": 83.1,         # Lab measurement (60° GU)
    "Drying_Time": 47.0    # Lab measurement (minutes)
}

val_table, overall_mae, overall_mape = evaluate_experimental_validation(
    ai_predicted=opt_results["predicted_properties"],
    experimental_measured=experimental_results
)

print("=" * 70)
print("EXPERIMENTAL VALIDATION REPORT: AI PREDICTION VS. BENCH MEASUREMENT")
print("=" * 70)
print(val_table.to_string(index=False))
print("-" * 70)
print(f"Overall Mean Absolute Error (MAE)      : {overall_mae:.2f}")
print(f"Overall Mean Absolute Percentage Error : {overall_mape:.2f}%")
print("=" * 70)

fig_val = plot_ai_vs_experimental_comparison(val_table, overall_mae, overall_mape)
plt.show()
